# Approximators

An **approximator** is the central object in BayesFlow. It wires together an adapter, an optional summary network, and an inference network, and exposes a unified API for training and inference.

Approximators are Keras models. They implement `.fit()` for training and task-specific inference methods (`.sample()`, `.log_prob()`, `.predict()`, etc.) for posterior queries.

This page covers:

1. [Approximator types](#approximator-types) — which one to use for your task
2. [Key methods](#key-methods) — `fit`, `sample`, `log_prob`, and friends
3. [Standalone training](#standalone-training) — using approximators without a workflow

## Approximator Types

| Approximator | Task | Primary inference method |
|---|---|---|
| `ContinuousApproximator` | Neural posterior / likelihood estimation (NPE, NLE) | `.sample()`, `.log_prob()` |
| `ModelComparisonApproximator` | Bayesian model selection (discrete posteriors) | `.predict()` |
| `RatioApproximator` | Likelihood-to-evidence ratio estimation (NRE-C) | `.log_ratio()` |
| `ScoringRuleApproximator` | Point estimation via scoring-rule minimization | `.estimate()` |
| `EnsembleApproximator` | Ensemble of heterogeneous approximators | `.sample()`, `.log_prob()`, `.estimate()` |
| `CompositionalApproximator` | Compositional / hierarchical distributions | `.compositional_sample()` |

All approximators live in `bayesflow.approximators` and are re-exported from the top-level `bayesflow` namespace.

## Key Methods

### `fit` and `compile`

All approximators share the same `fit` interface. You can pass either a simulator (BayesFlow builds the dataset for you) or a pre-built dataset:

```python
import keras
import bayesflow as bf

# Create an approximator (e.g., for a continuous distribution)
approximator = bf.ContinuousApproximator(inference_network=bf.networks.FlowMatching())

# Compile first — sets the optimizer; compatible with any keras optimizer
approximator.compile(optimizer="adam")

# Train via simulator (most common path)
history = approximator.fit(simulator=simulator, epochs=50, batch_size=128, num_batches=500)

# Or pass a pre-built dataset
dataset = approximator.build_dataset(simulator=simulator, batch_size=128, num_batches=500)

# Alternatively for data in memory
# dataset = bf.OfflineDataset(data_dict, batch_size=128)
history = approximator.fit(dataset=dataset, epochs=50)
```

`fit` returns a `keras.callbacks.History` object. All standard Keras callbacks work — early stopping, learning-rate schedules, TensorBoard, etc.:

```python
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=10),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5),
]
approximator.fit(simulator=simulator, epochs=200, callbacks=callbacks)
```

## ContinuousApproximator

The `ContinuousApproximator` is the standard choice for **neural posterior estimation (NPE)** and **neural likelihood estimation (NLE)**. It learns a continuous density $p(\theta \mid x)$ (or $p(x \mid \theta)$) using a normalizing flow or diffusion-based inference network.

```python
import bayesflow as bf

approximator = bf.approximators.ContinuousApproximator(
    inference_network=bf.networks.FlowMatching(),
    summary_network=bf.networks.SetTransformer(),  # optional
    adapter=adapter
)
```

### `.sample(num_samples, conditions)` — draw posterior samples

```python
# conditions: dict of observed data, preprocessed by the adapter
samples = approximator.sample(num_samples=1000, conditions={"x_obs": x_obs})
# returns: {"theta": np.ndarray of shape (num_conditions, 1000, theta_dim)}
```

### `.log_prob(data)` — evaluate the log-posterior density

```python
log_p = approximator.log_prob({"x_obs": x_obs, "theta": theta})
# returns: np.ndarray of shape (num_conditions,)
```

### `.ancestral_sample(conditions, ancestral_conditions)` — hierarchical sampling

For hierarchical models where child-level observations are conditioned on parent-level parameters:

```python
samples = approximator.ancestral_sample(
    conditions=child_conditions,
    ancestral_conditions=parent_conditions,
    num_samples=500
)
```

> **NPE vs NLE:** The choice between posterior estimation and likelihood estimation is controlled entirely by how the adapter routes variables — not by the approximator class. Point `inference_variables` at `theta` for NPE, or at `x` for NLE.

## ModelComparisonApproximator

Use this when comparing a discrete set of competing simulators / models. It learns posterior model probabilities $p(M_k \mid x)$ via a classifier trained with cross-entropy.

```python
approximator = bf.ModelComparisonApproximator(
    num_models=3,
    classifier_network=bf.networks.MLP(),
    summary_network=bf.networks.SetTransformer(),  # optional
    adapter=adapter,
)
```

Training requires a `ModelComparisonSimulator` (or a list of simulators):

```python
approximator.fit(simulators=[sim1, sim2, sim3], epochs=50, batch_size=128, num_batches=500)
```

### `.predict(conditions)` — posterior model probabilities

```python
probs = approximator.predict(conditions={"x_obs": x_obs})
# returns: np.ndarray of shape (num_conditions, num_models)
# probs[i, k] = estimated P(M_k | x_obs[i])
```

Pass `probs=False` to get raw logits instead.

## RatioApproximator

Implements **neural ratio estimation (NRE-C)**, learning the log likelihood-to-evidence ratio $\log \frac{p(x \mid \theta)}{p(x)}$ via contrastive learning. Useful when you need a flexible, simulation-based density ratio without committing to a specific parametric form.

```python
approximator = bf.RatioApproximator(
    inference_network=bf.networks.MLP(),
    adapter=adapter,
    K=5,        # number of negative samples per positive pair
    gamma=1.0   # odds of a dependent pair; increase to up-weight joint pairs
)
```

### `.log_ratio(data)` — estimated log likelihood-to-evidence ratio

```python
log_r = approximator.log_ratio({"x_obs": x_obs, "theta": theta})
# returns: Tensor of shape (num_conditions,)
```

## ScoringRuleApproximator

Minimizes a **proper scoring rule** (e.g., energy score, CRPS) instead of a likelihood. Useful when you want amortized point estimates or light-weight distributional summaries rather than full posterior samples.

```python
approximator = bf.ScoringRuleApproximator(
    inference_network=bf.networks.ScoringRuleNetwork(
        scoring_rules={
            "mean": bf.scoring_rules.MeanScore(), 
            "quantile": bf.scoring_rules.QuantileScore(q=[0.1, 0.5, 0.9])
        },
        subnet=bf.networks.MLP()
    ),
    adapter=adapter
)
```

### `.estimate(conditions)` — point estimates and distributional parameters

```python
estimates = approximator.estimate(conditions={"x_obs": x_obs})
# returns nested dict: {variable_name: {score_name: {head_name: np.ndarray}}}
# e.g., {"theta": {"energy": {"mean": ..., "std": ...}}}
```

Grouping can be changed with `groupby="score"` to inspect results per scoring rule.

## EnsembleApproximator

Wraps multiple named approximators and trains them jointly. It can wrap approximators of different types, such as point and distribution approximators. At inference time, it can return per-member outputs or merge them into a mixture.

```python
approximator = bf.EnsembleApproximator(
    approximators={
        "flow": bf.ContinuousApproximator(
            inference_network=bf.networks.FlowMatching(), adapter=adapter
        ),
        "diffusion": bf.ContinuousApproximator(
            inference_network=bf.networks.DiffusionModel(), adapter=adapter
        )
    }
)
```

All members share the same training data; the adapter is taken from the first member.

### `.sample(num_samples, conditions)` — merged mixture samples

```python
# Returns a single merged sample array by default (mixture of all members)
samples = approximator.sample(num_samples=500, conditions=conditions)

# Per-member results
samples = approximator.sample(num_samples=500, conditions=conditions, merge_members=False)
# returns: {"flow": ..., "diffusion": ...}

# Weighted mixture
samples = approximator.sample(
    num_samples=500, conditions=conditions,
    member_weights={"flow": 0.7, "diffusion": 0.3}
)
```

### `.log_prob(data)` — log-probability under the mixture

```python
log_p = approximator.log_prob(data, merge_members=True)  # log-sum-exp mixture
```

### `.estimate(conditions)` — point estimates (for members that support it)

```python
estimates = approximator.estimate(conditions=conditions)
# returns: {"member_name": {variable: array}} by default
# or groupby="variable": {variable: {"member_name": array}}
```

## CompositionalApproximator

Extends `ContinuousApproximator` to handle **compositional distributions** — settings where the inference conditions themselves have a compositional structure (e.g., multi-level or hierarchical observations processed jointly).

```python
approximator = bf.approximators.CompositionalApproximator(
    inference_network=bf.networks.FlowMatching(),
    adapter=adapter
)
```

### `.compositional_sample(num_samples, conditions)` — compositional posterior samples

Input conditions are expected to have shape `(n_datasets, n_compositional, ...)` where `n_compositional >= 2`.

```python
samples = approximator.compositional_sample(num_samples=500, conditions=conditions)
```

For most use cases, `ContinuousApproximator` is sufficient. Reach for `CompositionalApproximator` when your generative model has explicit compositional structure that must be preserved at inference time. See the [Compositional Diffusion example](../../examples/Compositional_Diffusion.ipynb) for a worked example.

## Standalone Training

Approximators are independent Keras models — you don't need a workflow to use them, even though workflows are recommended unless you need lower level control. This is useful when you want fine-grained control over the training loop, want to integrate BayesFlow into an existing codebase, or are building a non-standard pipeline.

### Minimal example

```python
import bayesflow as bf
import keras
import numpy as np

# 1. Define the adapter (maps simulator outputs to the right roles)
adapter = (
    bf.Adapter()
    .rename("x", "summary_variables")        # route to summary network before conditioning
    .rename("theta", "inference_variables")  # learn a posterior over the quantities theta
)

# 2. Build the approximator
approximator = bf.ContinuousApproximator(
    inference_network=bf.networks.FlowMatching(),
    summary_network=bf.networks.SetTransformer(),
    adapter=adapter
)

# 3. Compile
approximator.compile(optimizer=keras.optimizers.Adam(5e-4))

# 4. Train directly via a simulator
history = approximator.fit(
    simulator=simulator,  # or dataset=my_dataset for offline / disk training
    epochs=50,
    batch_size=128,
    num_batches=500
)
```

### Inspecting the training history

```python
from bayesflow.diagnostics.plots import loss

f = loss(history)
```

### Manual dataset control

If you need full control over the data pipeline — for example, to use an offline dataset or adjust worker counts — build the dataset explicitly:

```python
dataset = approximator.build_dataset(
    simulator=simulator,
    batch_size=128,
    num_batches=500,
    workers=4,
    use_multiprocessing=True
)

history = approximator.fit(dataset=dataset, epochs=50)
```

### Saving and loading

Standalone approximators are saved and loaded identically to workflow-managed ones:

```python
# Save
approximator.save("my_model.keras")

# Load (import bayesflow first to register custom objects)
import bayesflow as bf
import keras
approximator = keras.saving.load_model("my_model.keras")
```

See [Saving & Loading](saving_loading.ipynb) for details.